# Bayesian Inference for Cognitive Response Time Modelling
### Importance Sampling, Metropolis–Hastings MCMC, and Hamiltonian Monte Carlo

---

**Author:** *DIPON KONAR*  
**Language:** R  

---

## Overview

This portfolio applies Bayesian inference methods to two problems in cognitive science. In the first part, I model lexical decision response times using a mixture of lognormal distributions — the idea being that participants aren't always paying full attention, so we need to separate "engaged" responses from random/distracted ones. I test whether word frequency affects response time (the lexical-access hypothesis), using both Importance Sampling and Metropolis-Hastings MCMC and comparing the two.

In the second part, I implement Hamiltonian Monte Carlo from scratch on a simple Normal model and investigate how things like sample size, step size, and prior choice affect the posterior estimates.

---

### Table of Contents

**Part 1 — Mixture Model for Word Recognition**
- [1.1 Problem Statement](#1.1)
- [1.2 Model & Likelihood](#1.2)
- [1.3 Prior Specification](#1.3)
- [1.4 Data Exploration](#1.4)
- [1.5 Importance Sampling](#1.5)
- [1.6 Metropolis–Hastings MCMC](#1.6)
- [1.7 IS vs MCMC Comparison](#1.7)
- [1.8 Results & Interpretation](#1.8)

**Part 2 — Hamiltonian Monte Carlo**
- [2.1 Model & HMC Implementation](#2.1)
- [2.2 Baseline Posterior Estimation](#2.2)
- [2.3 Sensitivity to Sample Size](#2.3)
- [2.4 Sensitivity to Step-Size](#2.4)
- [2.5 Chain Diagnostics](#2.5)
- [2.6 Prior Sensitivity Analysis](#2.6)
- [2.7 Key Conclusions](#2.7)


---
## Setup

Loading the required libraries. I'm using `ggplot2` for all the plots, with `gridExtra` for arranging multiple plots together and `RColorBrewer` for palettes. Setting a seed so results are reproducible.


In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(gridExtra)
  library(RColorBrewer)
})

set.seed(42)

theme_portfolio <- function() {
  theme_bw(base_size = 13) +
  theme(
    plot.title       = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle    = element_text(size = 11, hjust = 0.5, colour = "grey40"),
    axis.title       = element_text(size = 12),
    panel.grid.minor = element_blank(),
    legend.position  = "top",
    legend.title     = element_blank(),
    strip.background = element_rect(fill = "grey92"),
    strip.text       = element_text(face = "bold")
  )
}

PALETTE <- c(IS = "#2166AC", MCMC = "#D6604D", True = "#1A9641")

cat("Environment ready.\n")

---
# Part 1 — Mixture Model for Word Recognition

<a id='1.1'></a>
## 1.1 Problem Statement

In a lexical decision task, people see a string of letters and decide as quickly as possible if it's a real word. The *lexical-access hypothesis* says that high-frequency words (like "the", "dog") should be recognised faster than low-frequency ones because they're more strongly activated in memory.

The problem is that response time distributions are right-skewed, and participants don't always pay full attention — sometimes they just respond randomly. A simple unimodal model won't capture this well, so we use a **mixture model** that explicitly includes an "inattentive" component.


<a id='1.2'></a>
## 1.2 Generative Model & Likelihood

Each trial has a latent indicator $Z_i \sim \text{Bernoulli}(\theta)$ — whether the participant was engaged or not. The response time is then drawn from one of two lognormal components:

$$
\text{rt}_i \sim
\begin{cases}
  \text{LogNormal}(\mu_i,\; \sigma) & \text{if } Z_i = 1 \quad \text{(engaged)}\\
  \text{LogNormal}(\gamma,\; \sigma) & \text{if } Z_i = 0 \quad \text{(distracted)}
\end{cases}
$$

where $\mu_i = \alpha + \beta \cdot \text{frequency}_i$.

The key parameter is $\beta$ — if it's negative, that supports the hypothesis (high-freq words are faster). Marginalising out $Z_i$, the observed log-likelihood becomes:

$$
\log L = \sum_i \log\!\left[
  \theta \cdot f_{\text{LN}}(\text{rt}_i;\, \alpha+\beta\,\text{freq}_i,\, \sigma)
  + (1-\theta) \cdot f_{\text{LN}}(\text{rt}_i;\, \gamma,\, \sigma)
\right]
$$


<a id='1.3'></a>
## 1.3 Prior Specification

$\sigma$ and $\alpha$ are fixed based on what's typical in the RT literature. The other three parameters get priors:

| Parameter | Prior | Why |
|-----------|-------|-----|
| $\beta$ | $\mathcal{N}(0, 0.5)$ | Weakly regularising, no strong assumption on direction |
| $\gamma$ | $\mathcal{N}(5, 0.5)$ | Centred around fast random responses |
| $\theta$ | $\text{Beta}(70, 30)$ | Strong prior that engagement is high (~70%) |

The Beta(70,30) prior on $\theta$ is fairly informative — it reflects that participants are probably paying attention most of the time.


<a id='1.4'></a>
## 1.4 Data Exploration

Loading the dataset and doing a quick look at the RT distributions by frequency group. I expected the high-frequency distribution to be shifted left (faster), let's see if that's actually the case.


In [ ]:
dat <- read.csv("word-recognition.csv")

# Tidy column names and encode frequency as a labelled factor for plotting
dat$freq_label <- factor(
  dat$frequency,
  levels = c(0, 1),
  labels = c("Low Frequency", "High Frequency")
)

cat(sprintf("Dataset: %d trials | %d high-frequency | %d low-frequency\n",
            nrow(dat),
            sum(dat$frequency == 1),
            sum(dat$frequency == 0)))
cat(sprintf("RT range: %.1f – %.1f ms | Median: %.1f ms\n",
            min(dat$rt), max(dat$rt), median(dat$rt)))

In [ ]:
rt_summary <- aggregate(rt ~ freq_label, data = dat, FUN = function(x)
  c(mean = mean(x), median = median(x), sd = sd(x), n = length(x)))
print(do.call(data.frame, rt_summary))

p_eda <- ggplot(dat, aes(x = rt, fill = freq_label, colour = freq_label)) +
  geom_density(alpha = 0.35, linewidth = 0.9) +
  geom_rug(alpha = 0.25, linewidth = 0.4) +
  scale_fill_manual(values  = c("Low Frequency"  = "#D6604D",
                                "High Frequency" = "#2166AC")) +
  scale_colour_manual(values = c("Low Frequency"  = "#D6604D",
                                 "High Frequency" = "#2166AC")) +
  labs(
    title    = "Response Time Distributions by Word Frequency",
    subtitle = "Right skew motivates the lognormal component in the mixture model",
    x        = "Response Time (ms)",
    y        = "Density"
  ) +
  theme_portfolio()

print(p_eda)

<a id='1.5'></a>
## 1.5 Importance Sampling

Importance sampling works by drawing from a proposal distribution $q(\boldsymbol{\psi})$ instead of the posterior directly, then reweighting samples using:

$$w_i = \frac{L(\boldsymbol{\psi}^{(i)} \mid \mathbf{y}) \cdot p(\boldsymbol{\psi}^{(i)})}{q(\boldsymbol{\psi}^{(i)})}$$

The effective sample size (ESS) tells you how well the proposal covered the posterior — if it's very low, the weights are degenerate and the estimate is unreliable. I used Normal proposals for $\beta$ and $\gamma$, and Uniform for $\theta$.


In [ ]:
SIGMA_FIXED <- 0.4
ALPHA_FIXED <- 6.0

#
# Numerically stable via log-sum-exp with a small floor to prevent log(0).
log_likelihood_mixture <- function(beta, gamma, theta, data,
                                   sigma = SIGMA_FIXED,
                                   alpha = ALPHA_FIXED) {
  mu_i      <- alpha + beta * data$frequency
  lik_vals  <- theta       * dlnorm(data$rt, meanlog = mu_i,  sdlog = sigma) +
               (1 - theta) * dlnorm(data$rt, meanlog = gamma, sdlog = sigma)
  sum(log(lik_vals + 1e-300))
}

log_prior_mixture <- function(beta, gamma, theta) {
  dnorm(beta,  mean = 0, sd = 0.5,  log = TRUE) +
  dnorm(gamma, mean = 5, sd = 0.5,  log = TRUE) +
  dbeta(theta, shape1 = 70, shape2 = 30, log = TRUE)
}

#
# Proposals: beta ~ N(0,1), gamma ~ N(5,1), theta ~ Uniform(0,1)
# Using a wider proposal than the prior for better tail coverage.
run_importance_sampling <- function(data, N = 10000, seed = 42) {
  set.seed(seed)

  prop_beta  <- rnorm(N, mean = 0, sd = 1)
  prop_gamma <- rnorm(N, mean = 5, sd = 1)
  prop_theta <- runif(N, min = 0, max = 1)

  log_weights <- mapply(
    function(b, g, t) {
      ll   <- log_likelihood_mixture(b, g, t, data)
      lp   <- log_prior_mixture(b, g, t)
      lq   <- dnorm(b, 0, 1, log = TRUE) +
              dnorm(g, 5, 1, log = TRUE) +
              dunif(t, 0, 1, log = TRUE)
      ll + lp - lq
    },
    prop_beta, prop_gamma, prop_theta
  )

  # Numerically stable weight normalisation via log-sum-exp
  log_weights[!is.finite(log_weights)] <- -Inf
  weights <- exp(log_weights - max(log_weights))
  weights <- weights / sum(weights)

  # Effective sample size diagnostic
  ess <- 1 / sum(weights^2)
  cat(sprintf("IS Effective Sample Size: %.0f / %d (%.1f%%)\n",
              ess, N, 100 * ess / N))

  # Resample N/2 draws proportional to weights
  idx <- sample(N, size = N / 2, replace = TRUE, prob = weights)

  data.frame(
    beta  = prop_beta[idx],
    gamma = prop_gamma[idx],
    theta = prop_theta[idx]
  )
}

post_is <- run_importance_sampling(dat, N = 10000)

cat(sprintf("\nIS Posterior Summaries:\n"))
cat(sprintf("  beta  : mean = %.4f, sd = %.4f, 95%% CI = [%.4f, %.4f]\n",
            mean(post_is$beta), sd(post_is$beta),
            quantile(post_is$beta, 0.025), quantile(post_is$beta, 0.975)))
cat(sprintf("  gamma : mean = %.4f, sd = %.4f\n",
            mean(post_is$gamma), sd(post_is$gamma)))
cat(sprintf("  theta : mean = %.4f, sd = %.4f\n",
            mean(post_is$theta), sd(post_is$theta)))

<a id='1.6'></a>
## 1.6 Metropolis–Hastings MCMC

MH builds a Markov chain that converges to the target posterior. At each step, a new proposal is made and accepted/rejected using the ratio:

$$\alpha(\boldsymbol{\psi}^*, \boldsymbol{\psi}^{(t)}) = \min\left(1,\; \frac{p(\boldsymbol{\psi}^* \mid \mathbf{y})}{p(\boldsymbol{\psi}^{(t)} \mid \mathbf{y})}\right)$$

Since $\theta \in [0,1]$, I reflect proposals that go outside this range back into the valid domain. The first 5000 samples are discarded as burn-in.


In [ ]:
#
# Random-walk proposal with reflected boundary for theta.
run_mh_mcmc <- function(data,
                        n_iter       = 10000,
                        n_burn       = 5000,
                        proposal_sd  = 0.05,
                        init         = c(beta = 0, gamma = 5, theta = 0.7),
                        seed         = 42) {
  set.seed(seed)

  chain      <- matrix(NA_real_, nrow = n_iter, ncol = 3,
                       dimnames = list(NULL, c("beta", "gamma", "theta")))
  chain[1, ] <- init
  n_accept   <- 0L

  log_post <- function(beta, gamma, theta) {
    if (theta <= 0 || theta >= 1) return(-Inf)
    log_likelihood_mixture(beta, gamma, theta, data) +
    log_prior_mixture(beta, gamma, theta)
  }

  lp_current <- log_post(chain[1, "beta"], chain[1, "gamma"], chain[1, "theta"])

  for (i in seq(2, n_iter)) {
    proposal        <- chain[i - 1, ] + rnorm(3, 0, proposal_sd)
    proposal["theta"] <- pmax(0.001, pmin(0.999, proposal["theta"]))  # reflect

    lp_proposed <- log_post(proposal["beta"], proposal["gamma"], proposal["theta"])
    log_alpha   <- lp_proposed - lp_current

    if (is.finite(log_alpha) && log(runif(1)) < log_alpha) {
      chain[i, ] <- proposal
      lp_current <- lp_proposed
      n_accept   <- n_accept + 1L
    } else {
      chain[i, ] <- chain[i - 1, ]
    }
  }

  accept_rate <- n_accept / n_iter
  cat(sprintf("MH acceptance rate: %.1f%%\n", 100 * accept_rate))
  if (accept_rate < 0.1 || accept_rate > 0.6)
    warning("Acceptance rate outside [10%, 60%] — consider tuning proposal_sd.")

  post  <- as.data.frame(chain[(n_burn + 1):n_iter, ])
  full  <- as.data.frame(chain)
  list(posterior = post, full_chain = full, accept_rate = accept_rate)
}

mcmc_result  <- run_mh_mcmc(dat, n_iter = 10000, n_burn = 5000,
                             proposal_sd = 0.05)
post_mcmc    <- mcmc_result$posterior

cat(sprintf("\nMCMC Posterior Summaries:\n"))
cat(sprintf("  beta  : mean = %.4f, sd = %.4f, 95%% CI = [%.4f, %.4f]\n",
            mean(post_mcmc$beta), sd(post_mcmc$beta),
            quantile(post_mcmc$beta, 0.025), quantile(post_mcmc$beta, 0.975)))
cat(sprintf("  gamma : mean = %.4f, sd = %.4f\n",
            mean(post_mcmc$gamma), sd(post_mcmc$gamma)))
cat(sprintf("  theta : mean = %.4f, sd = %.4f\n",
            mean(post_mcmc$theta), sd(post_mcmc$theta)))

In [ ]:
full_chain <- mcmc_result$full_chain
full_chain$iter   <- seq_len(nrow(full_chain))
full_chain$phase  <- ifelse(full_chain$iter <= 5000, "Burn-in", "Post burn-in")

make_traceplot <- function(df, param, ylabel, title) {
  ggplot(df, aes_string(x = "iter", y = param, colour = "phase")) +
    geom_line(linewidth = 0.4, alpha = 0.8) +
    scale_colour_manual(values = c("Burn-in" = "grey60", "Post burn-in" = "#2166AC")) +
    labs(title = title, x = "Iteration", y = ylabel) +
    theme_portfolio() +
    theme(legend.position = "none")
}

p_tr1 <- make_traceplot(full_chain, "beta",  expression(beta),  "Traceplot: β")
p_tr2 <- make_traceplot(full_chain, "gamma", expression(gamma), "Traceplot: γ")
p_tr3 <- make_traceplot(full_chain, "theta", expression(theta), "Traceplot: θ")

grid.arrange(p_tr1, p_tr2, p_tr3, ncol = 3,
             top = "MCMC Chain Traceplots (grey = burn-in, blue = retained)")

In [ ]:
make_density_plot <- function(samples, param, xlabel, title, vline = NULL) {
  df <- data.frame(x = samples[[param]])
  p  <- ggplot(df, aes(x = x)) +
    geom_density(fill = "#2166AC", alpha = 0.25, colour = "#2166AC", linewidth = 1) +
    geom_vline(xintercept = mean(samples[[param]]),
               linetype = "dashed", colour = "#2166AC", linewidth = 0.9) +
    labs(title = title, x = xlabel, y = "Density") +
    theme_portfolio()
  if (!is.null(vline))
    p <- p + geom_vline(xintercept = vline, colour = "#D6604D",
                        linetype = "dotted", linewidth = 0.9)
  p
}

p_d1 <- make_density_plot(post_mcmc, "beta",  expression(beta),
                           "Posterior: β", vline = 0)
p_d2 <- make_density_plot(post_mcmc, "gamma", expression(gamma), "Posterior: γ")
p_d3 <- make_density_plot(post_mcmc, "theta", expression(theta), "Posterior: θ")

grid.arrange(p_d1, p_d2, p_d3, ncol = 3,
             top = "MCMC Marginal Posterior Densities (dashed = posterior mean, dotted red = β=0)")

<a id='1.7'></a>
## 1.7 Posterior Comparison: IS vs MCMC

If both methods are implemented correctly, they should give similar posteriors since they're targeting the same distribution. This is a useful sanity check — IS tends to have more variance in the tails because of weight degeneracy issues, while MCMC can have autocorrelation problems if the chain mixes slowly.


In [ ]:
build_comparison_df <- function(is_samples, mcmc_samples, param) {
  rbind(
    data.frame(value = is_samples[[param]],   method = "Importance Sampling"),
    data.frame(value = mcmc_samples[[param]], method = "MCMC (M-H)")
  )
}

make_comparison_density <- function(df, xlabel, title) {
  ggplot(df, aes(x = value, fill = method, colour = method)) +
    geom_density(alpha = 0.30, linewidth = 1) +
    scale_fill_manual(values   = c("Importance Sampling" = "#2166AC",
                                   "MCMC (M-H)"          = "#D6604D")) +
    scale_colour_manual(values = c("Importance Sampling" = "#2166AC",
                                   "MCMC (M-H)"          = "#D6604D")) +
    labs(title = title, x = xlabel, y = "Density") +
    theme_portfolio()
}

p_cmp1 <- make_comparison_density(
  build_comparison_df(post_is, post_mcmc, "beta"),
  expression(beta), "Posterior Comparison: β")

p_cmp2 <- make_comparison_density(
  build_comparison_df(post_is, post_mcmc, "gamma"),
  expression(gamma), "Posterior Comparison: γ")

p_cmp3 <- make_comparison_density(
  build_comparison_df(post_is, post_mcmc, "theta"),
  expression(theta), "Posterior Comparison: θ")

grid.arrange(p_cmp1, p_cmp2, p_cmp3, ncol = 3,
             top = "IS vs MCMC: Marginal Posterior Density Comparison")

<a id='1.8'></a>
## 1.8 Results & Interpretation

Computing posterior summaries for $\beta$ (the frequency effect) and checking how much posterior mass falls below/above zero.


In [ ]:
summarise_posterior <- function(samples, label) {
  b <- samples$beta
  cat(sprintf("\n%s\n", label))
  cat(sprintf("  Mean β       : %+.4f\n",       mean(b)))
  cat(sprintf("  Median β     : %+.4f\n",       median(b)))
  cat(sprintf("  SD           :  %.4f\n",       sd(b)))
  cat(sprintf("  95%% CrI      : [%+.4f, %+.4f]\n",
              quantile(b, 0.025), quantile(b, 0.975)))
  cat(sprintf("  P(β < 0)     :  %.4f\n",       mean(b < 0)))
  cat(sprintf("  P(β > 0)     :  %.4f\n",       mean(b > 0)))
}

summarise_posterior(post_mcmc, "MCMC Posterior — β (frequency effect)")
summarise_posterior(post_is,   "IS Posterior   — β (frequency effect)")

cat(sprintf("\nMCMC Posterior — θ (task engagement)\n"))
cat(sprintf("  Mean θ      : %.4f\n",    mean(post_mcmc$theta)))
cat(sprintf("  95%% CrI     : [%.4f, %.4f]\n",
            quantile(post_mcmc$theta, 0.025),
            quantile(post_mcmc$theta, 0.975)))

In [ ]:
df_beta <- data.frame(beta = post_mcmc$beta)
ci_lo   <- quantile(post_mcmc$beta, 0.025)
ci_hi   <- quantile(post_mcmc$beta, 0.975)

ggplot(df_beta, aes(x = beta)) +
  geom_histogram(aes(y = after_stat(density)), bins = 45,
                 fill = "#AEC6CF", colour = "white", linewidth = 0.3) +
  geom_density(colour = "#2166AC", linewidth = 1.1) +
  geom_vline(xintercept = 0,               colour = "#D6604D",  linewidth = 1,   linetype = "dashed") +
  geom_vline(xintercept = mean(df_beta$beta), colour = "#2166AC",  linewidth = 0.9, linetype = "dotdash") +
  geom_vline(xintercept = ci_lo,           colour = "grey40",   linewidth = 0.8, linetype = "dotted") +
  geom_vline(xintercept = ci_hi,           colour = "grey40",   linewidth = 0.8, linetype = "dotted") +
  annotate("text", x = 0.015, y = Inf, label = "β = 0", vjust = 1.6,
           colour = "#D6604D", size = 3.8, fontface = "italic") +
  labs(
    title    = "Posterior Distribution of the Frequency Effect (β)",
    subtitle = "Red dashed: null hypothesis  |  Blue dash-dot: posterior mean  |  Grey dotted: 95% CrI",
    x        = expression(beta ~ "(frequency effect on log-RT)"),
    y        = "Density"
  ) +
  theme_portfolio()

### Interpretation

**Frequency effect ($\beta$):**  
Interestingly, the posterior of $\beta$ concentrates on *positive* values — meaning higher-frequency words were actually associated with *longer* RTs in this sample, which is the opposite of what the lexical-access hypothesis predicts. That said, this doesn't necessarily disprove the hypothesis. You'd really need a proper model comparison (e.g., Bayes factors) to make that call — the posterior of $\beta$ just tells us what the model estimates given the data, not whether the hypothesis is supported.

**Task engagement ($\theta$):**  
The posterior for $\theta$ sits around 0.71, suggesting the participant was engaged on roughly 71% of trials. The prior was already centred at 0.70, so the data don't shift this much.

**IS vs MCMC:**  
Both methods give broadly similar results, which is reassuring. There are some differences in the tails of $\beta$, likely because IS can have weight degeneracy problems. For models like this, MCMC is generally more reliable.


---
# Part 2 — Hamiltonian Monte Carlo

<a id='2.1'></a>
## 2.1 Model & HMC Implementation

HMC improves on standard random-walk MCMC by using gradient information to make smarter, larger proposals. It works by introducing auxiliary momentum variables and simulating Hamiltonian dynamics:

$$H(\mathbf{q}, \mathbf{p}) = \underbrace{-\log p(\mathbf{q} \mid \mathbf{y})}_{V(\mathbf{q})} + \underbrace{\frac{1}{2}\mathbf{p}^\top \mathbf{p}}_{K(\mathbf{p})}$$

The dynamics are approximated using leapfrog integration, and a Metropolis correction handles any numerical error.

**Model:** $y_i \sim \mathcal{N}(\mu, \sigma^2)$ with 500 observations from $\mathcal{N}(800, 100)$.  
**Priors:** $\mu \sim \mathcal{N}(1000, 20)$, $\sigma \sim \mathcal{N}(10, 2)$.


In [ ]:
set.seed(42)
TRUE_MU  <- 800
TRUE_VAR <- 100
y_hmc    <- rnorm(500, mean = TRUE_MU, sd = sqrt(TRUE_VAR))
n_hmc    <- length(y_hmc)

cat(sprintf("Data: n=%d | sample mean=%.2f | sample SD=%.2f\n",
            n_hmc, mean(y_hmc), sd(y_hmc)))

ggplot(data.frame(y = y_hmc), aes(x = y)) +
  geom_histogram(aes(y = after_stat(density)), bins = 40,
                 fill = "#AEC6CF", colour = "white", linewidth = 0.3) +
  geom_density(colour = "#2166AC", linewidth = 1.1) +
  geom_vline(xintercept = TRUE_MU, colour = "#D6604D",
             linetype = "dashed", linewidth = 1) +
  annotate("text", x = TRUE_MU + 4, y = Inf, label = paste("True μ =", TRUE_MU),
           vjust = 2, colour = "#D6604D", size = 3.8) +
  labs(title = "Synthetic Normal Data",
       subtitle = "500 draws from N(800, 100)",
       x = "y", y = "Density") +
  theme_portfolio()

In [ ]:
#
# grad_mu    = dV/dμ    (from Normal likelihood + Normal prior on μ)
# grad_sigma = dV/dσ    (from Normal likelihood + Normal prior on σ)
gradient_normal <- function(mu, sigma, y, n, m, s, a, b) {
  grad_mu    <- ((n * mu - sum(y)) / sigma^2) + ((mu - m) / s^2)
  grad_sigma <- (n / sigma) - (sum((y - mu)^2) / sigma^3) + ((sigma - a) / b^2)
  c(grad_mu, grad_sigma)
}

potential_energy <- function(mu, sigma, y, n, m, s, a, b) {
  if (sigma <= 0) return(Inf)
  -( sum(dnorm(y, mu, sigma, log = TRUE)) +
     dnorm(mu, m, s, log = TRUE) +
     dnorm(sigma, a, b, log = TRUE) )
}

#
# Arguments:
#   y, n        — data and sample size
#   m, s        — prior parameters for μ  (Normal)
#   a, b        — prior parameters for σ  (Normal)
#   step        — leapfrog step-size ε
#   L           — number of leapfrog steps per proposal
#   initial_q   — starting position c(mu, sigma)
#   nsamp       — total samples to generate (including burn-in)
#   nburn       — number of warm-up samples to discard
#
# Returns a data.frame with columns mu_chain, sigma_chain.
hmc_sampler <- function(y, n, m, s, a, b,
                        step, L, initial_q, nsamp, nburn) {
  mu_chain    <- numeric(nsamp)
  sigma_chain <- numeric(nsamp)
  n_reject    <- 0L

  mu_chain[1]    <- initial_q[1]
  sigma_chain[1] <- initial_q[2]

  i <- 1L
  while (i < nsamp) {
    q         <- c(mu_chain[i], sigma_chain[i])
    p         <- rnorm(2, 0, 1)            # sample fresh momentum
    current_q <- q
    current_p <- p

    current_V <- potential_energy(q[1], q[2], y, n, m, s, a, b)
    current_K <- sum(p^2) / 2

    for (l in seq_len(L)) {
      p <- p - (step / 2) * gradient_normal(q[1], q[2], y, n, m, s, a, b)
      q <- q + step * p
      p <- p - (step / 2) * gradient_normal(q[1], q[2], y, n, m, s, a, b)
    }

    proposed_V   <- potential_energy(q[1], q[2], y, n, m, s, a, b)
    proposed_K   <- sum(p^2) / 2
    accept_prob  <- min(1, exp(current_V + current_K - proposed_V - proposed_K))

    if (is.finite(accept_prob) && runif(1) < accept_prob) {
      mu_chain[i + 1]    <- q[1]
      sigma_chain[i + 1] <- q[2]
    } else {
      mu_chain[i + 1]    <- current_q[1]
      sigma_chain[i + 1] <- current_q[2]
      n_reject <- n_reject + 1L
    }
    i <- i + 1L
  }

  accept_rate <- 1 - n_reject / nsamp
  cat(sprintf("HMC acceptance rate: %.1f%%\n", 100 * accept_rate))

  full_chain <- data.frame(mu_chain, sigma_chain)
  full_chain[(nburn + 1):nsamp, ]
}

cat("HMC sampler defined.\n")

<a id='2.2'></a>
## 2.2 Baseline Posterior Estimation

Running the HMC sampler with `step = 0.02`, `L = 12`, 6000 samples and 2000 burn-in. The true parameter values are $\mu = 800$ and $\sigma = 10$, so we can directly check whether the sampler recovers them.


In [ ]:
set.seed(42)
df_baseline <- hmc_sampler(
  y = y_hmc, n = n_hmc,
  m = 1000, s = 20, a = 10, b = 2,
  step = 0.02, L = 12,
  initial_q = c(1000, 11),
  nsamp = 6000, nburn = 2000
)

cat(sprintf("\nBaseline HMC Posterior Summaries:\n"))
cat(sprintf("  μ : mean = %.2f | SD = %.2f | 95%% CrI = [%.2f, %.2f] | true = %d\n",
            mean(df_baseline$mu_chain), sd(df_baseline$mu_chain),
            quantile(df_baseline$mu_chain, 0.025),
            quantile(df_baseline$mu_chain, 0.975), TRUE_MU))
cat(sprintf("  σ : mean = %.2f | SD = %.2f | 95%% CrI = [%.2f, %.2f] | true = %d\n",
            mean(df_baseline$sigma_chain), sd(df_baseline$sigma_chain),
            quantile(df_baseline$sigma_chain, 0.025),
            quantile(df_baseline$sigma_chain, 0.975), as.integer(sqrt(TRUE_VAR))))

In [ ]:
make_hmc_density <- function(samples, col_name, true_val, xlabel, title) {
  df <- data.frame(x = samples[[col_name]])
  ggplot(df, aes(x = x)) +
    geom_histogram(aes(y = after_stat(density)), bins = 40,
                   fill = "#AEC6CF", colour = "white", linewidth = 0.2) +
    geom_density(colour = "#2166AC", linewidth = 1.1) +
    geom_vline(xintercept = true_val, colour = "#D6604D",
               linetype = "dashed", linewidth = 1) +
    geom_vline(xintercept = mean(samples[[col_name]]), colour = "#2166AC",
               linetype = "dotdash", linewidth = 0.9) +
    annotate("text", x = true_val, y = Inf,
             label = paste("True =", true_val),
             vjust = 2.2, hjust = -0.1, colour = "#D6604D", size = 3.5) +
    labs(title = title, x = xlabel, y = "Density") +
    theme_portfolio()
}

p_mu_base    <- make_hmc_density(df_baseline, "mu_chain",    TRUE_MU,
                                 expression(mu),    "Posterior of μ")
p_sigma_base <- make_hmc_density(df_baseline, "sigma_chain", sqrt(TRUE_VAR),
                                 expression(sigma), "Posterior of σ")

grid.arrange(p_mu_base, p_sigma_base, ncol = 2,
             top = "HMC Baseline Posteriors (red dashed = true value, blue dash-dot = posterior mean)")

<a id='2.3'></a>
## 2.3 Sensitivity to Sample Size

Here I'm checking what happens when you run the sampler with fewer samples. I'd expect that with very few draws, the posterior estimate will be noisy and unstable, and more samples should give a smoother, more reliable result.


In [ ]:
nsamp_values <- c(100, 1000, 6000)

results_nsamp <- lapply(nsamp_values, function(ns) {
  set.seed(42)
  nb <- round(ns / 3)
  df <- hmc_sampler(y = y_hmc, n = n_hmc,
                    m = 1000, s = 20, a = 10, b = 2,
                    step = 0.02, L = 12,
                    initial_q = c(1000, 11),
                    nsamp = ns, nburn = nb)
  df$nsamp_label <- paste0("nsamp = ", ns)
  df
})

df_nsamp_all <- do.call(rbind, results_nsamp)
df_nsamp_all$nsamp_label <- factor(df_nsamp_all$nsamp_label,
                                   levels = paste0("nsamp = ", nsamp_values))

p_ns_mu <- ggplot(df_nsamp_all, aes(x = mu_chain)) +
  geom_density(fill = "#2166AC", alpha = 0.25, colour = "#2166AC", linewidth = 0.9) +
  geom_vline(xintercept = TRUE_MU, colour = "#D6604D",
             linetype = "dashed", linewidth = 0.9) +
  facet_wrap(~ nsamp_label, scales = "free") +
  labs(title    = "Posterior of μ Across Sample Sizes",
       subtitle = "Red dashed = true μ = 800",
       x = expression(mu), y = "Density") +
  theme_portfolio()

p_ns_sigma <- ggplot(df_nsamp_all, aes(x = sigma_chain)) +
  geom_density(fill = "#D6604D", alpha = 0.25, colour = "#D6604D", linewidth = 0.9) +
  geom_vline(xintercept = sqrt(TRUE_VAR), colour = "#1A9641",
             linetype = "dashed", linewidth = 0.9) +
  facet_wrap(~ nsamp_label, scales = "free") +
  labs(title    = "Posterior of σ Across Sample Sizes",
       subtitle = "Green dashed = true σ = 10",
       x = expression(sigma), y = "Density") +
  theme_portfolio()

grid.arrange(p_ns_mu, p_ns_sigma, nrow = 2)

### Interpretation

With only 100 total samples the estimates are quite rough — the chain hasn't seen enough of the posterior yet. At 1000 samples things improve but it's still a bit bumpy in places. At 6000 samples the posteriors look smooth and well-concentrated around the true values. So as expected, you need a reasonable number of samples for MCMC to work well — there's a clear trade-off between computation time and Monte Carlo error.


<a id='2.4'></a>
## 2.4 Sensitivity to Step-Size

The leapfrog step-size $\varepsilon$ matters a lot for HMC:
- Too small: the chain barely moves each step, so it takes forever to explore the posterior
- Too large: integration becomes inaccurate and the acceptance rate drops

I'm testing $\varepsilon \in \{0.001, 0.005, 0.02\}$ to see this effect in practice.


In [ ]:
step_values <- c(0.001, 0.005, 0.02)

results_step <- lapply(step_values, function(st) {
  set.seed(42)
  df <- hmc_sampler(y = y_hmc, n = n_hmc,
                    m = 1000, s = 20, a = 10, b = 2,
                    step = st, L = 12,
                    initial_q = c(1000, 11),
                    nsamp = 6000, nburn = 2000)
  df$step_label <- paste0("ε = ", st)
  df
})

# Summary table
cat("\nPosterior Means by Step-Size:\n")
cat(sprintf("  %-10s  %-12s  %-12s\n", "Step (ε)", "Mean μ", "Mean σ"))
cat(rep("-", 40), "\n", sep = "")
for (i in seq_along(step_values)) {
  cat(sprintf("  %-10s  %-12.2f  %-12.2f\n",
              step_values[i],
              mean(results_step[[i]]$mu_chain),
              mean(results_step[[i]]$sigma_chain)))
}

df_step_all <- do.call(rbind, results_step)
df_step_all$step_label <- factor(df_step_all$step_label,
                                 levels = paste0("ε = ", step_values))

p_st_mu <- ggplot(df_step_all, aes(x = mu_chain)) +
  geom_density(fill = "#2166AC", alpha = 0.25, colour = "#2166AC", linewidth = 0.9) +
  geom_vline(xintercept = TRUE_MU, colour = "#D6604D",
             linetype = "dashed", linewidth = 0.9) +
  facet_wrap(~ step_label, scales = "free") +
  labs(title = "Posterior of μ Across Step-Sizes",
       subtitle = "Red dashed = true μ = 800",
       x = expression(mu), y = "Density") +
  theme_portfolio()

p_st_sigma <- ggplot(df_step_all, aes(x = sigma_chain)) +
  geom_density(fill = "#D6604D", alpha = 0.25, colour = "#D6604D", linewidth = 0.9) +
  geom_vline(xintercept = sqrt(TRUE_VAR), colour = "#1A9641",
             linetype = "dashed", linewidth = 0.9) +
  facet_wrap(~ step_label, scales = "free") +
  labs(title = "Posterior of σ Across Step-Sizes",
       subtitle = "Green dashed = true σ = 10",
       x = expression(sigma), y = "Density") +
  theme_portfolio()

grid.arrange(p_st_mu, p_st_sigma, nrow = 2)

### Interpretation

At $\varepsilon = 0.001$ the chain barely moves and doesn't converge to the right region within the sample budget — the posterior mean of $\mu$ ends up around 989 instead of 800. At $\varepsilon = 0.005$ things are better but still off. At $\varepsilon = 0.02$ the chain converges properly.

This was a bit surprising to me — I thought smaller steps would always be safer, but clearly too small is just as bad because the chain can't explore enough in the given number of steps.


<a id='2.5'></a>
## 2.5 Chain Diagnostics

Looking at traceplots to check that the well-tuned chain ($\varepsilon = 0.02$) is actually mixing properly. A good chain should look like a "fuzzy caterpillar" — no trends, no getting stuck, stable variance throughout. Also plotting ACFs to check autocorrelation.


In [ ]:
df_trace <- results_step[[3]]   # ε = 0.02
df_trace$iter <- seq_len(nrow(df_trace))

p_trace_mu <- ggplot(df_trace, aes(x = iter, y = mu_chain)) +
  geom_line(colour = "#2166AC", alpha = 0.6, linewidth = 0.35) +
  geom_hline(yintercept = mean(df_trace$mu_chain),
             colour = "#D6604D", linetype = "dashed", linewidth = 0.9) +
  labs(title = "Traceplot: μ  (ε = 0.02)",
       subtitle = "Red dashed = posterior mean",
       x = "Post-Burn-In Iteration", y = expression(mu)) +
  theme_portfolio()

p_trace_sigma <- ggplot(df_trace, aes(x = iter, y = sigma_chain)) +
  geom_line(colour = "#D6604D", alpha = 0.6, linewidth = 0.35) +
  geom_hline(yintercept = mean(df_trace$sigma_chain),
             colour = "#2166AC", linetype = "dashed", linewidth = 0.9) +
  labs(title = "Traceplot: σ  (ε = 0.02)",
       subtitle = "Blue dashed = posterior mean",
       x = "Post-Burn-In Iteration", y = expression(sigma)) +
  theme_portfolio()

grid.arrange(p_trace_mu, p_trace_sigma, nrow = 2,
             top = "HMC Chain Diagnostics — Well-Tuned Sampler (step = 0.02, L = 12)")

In [ ]:
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
acf(df_trace$mu_chain,    main = expression("ACF: " * mu),    lag.max = 50,
    col = "#2166AC", lwd = 1.5)
acf(df_trace$sigma_chain, main = expression("ACF: " * sigma), lag.max = 50,
    col = "#D6604D", lwd = 1.5)
par(mfrow = c(1, 1))

### Interpretation

The traceplots look good — both chains are moving freely around their posterior means with no obvious drift. The ACFs drop off quickly to near zero, which shows the samples are approximately independent. This is one of the main advantages of HMC over random-walk MCMC, which tends to have much higher autocorrelation.


<a id='2.6'></a>
## 2.6 Prior Sensitivity Analysis

I want to check how much the prior choice for $\mu$ affects the posterior. If the prior is well-specified or vague, the likelihood should dominate. If the prior is tight and wrong, it could bias the posterior substantially.

Testing different combinations of prior mean ($m \in \{400, 1000\}$) and prior SD ($s \in \{5, 20, 100\}$).


In [ ]:
prior_configs <- list(
  list(m = 400,  s =   5,  label = "μ ~ N(400, 5)"),
  list(m = 400,  s =  20,  label = "μ ~ N(400, 20)"),
  list(m = 1000, s =   5,  label = "μ ~ N(1000, 5)"),
  list(m = 1000, s =  20,  label = "μ ~ N(1000, 20)"),
  list(m = 1000, s = 100,  label = "μ ~ N(1000, 100)")
)

results_prior <- lapply(prior_configs, function(cfg) {
  set.seed(42)
  df <- hmc_sampler(y = y_hmc, n = n_hmc,
                    m = cfg$m, s = cfg$s, a = 10, b = 2,
                    step = 0.02, L = 12,
                    initial_q = c(1000, 11),
                    nsamp = 6000, nburn = 2000)
  df$prior_label <- cfg$label
  df
})

# Summary table
cat("\nPosterior Summaries by Prior Specification:\n")
cat(sprintf("  %-22s  %-10s  %-10s  %-24s\n",
            "Prior", "Mean μ", "SD μ", "95% CrI"))
cat(rep("-", 75), "\n", sep = "")
for (i in seq_along(prior_configs)) {
  mu_samples <- results_prior[[i]]$mu_chain
  cat(sprintf("  %-22s  %-10.2f  %-10.2f  [%.2f, %.2f]\n",
              prior_configs[[i]]$label,
              mean(mu_samples), sd(mu_samples),
              quantile(mu_samples, 0.025),
              quantile(mu_samples, 0.975)))
}

In [ ]:
df_prior_all <- do.call(rbind, results_prior)
label_order  <- sapply(prior_configs, `[[`, "label")
df_prior_all$prior_label <- factor(df_prior_all$prior_label, levels = label_order)

# Colour-code by prior mean
df_prior_all$prior_mean_grp <- ifelse(
  grepl("400", df_prior_all$prior_label),
  "Prior mean = 400 (misspecified)",
  "Prior mean = 1000 (near truth)"
)

ggplot(df_prior_all, aes(x = mu_chain, fill = prior_mean_grp, colour = prior_mean_grp)) +
  geom_density(alpha = 0.30, linewidth = 0.9) +
  geom_vline(xintercept = TRUE_MU, colour = "black",
             linetype = "dashed", linewidth = 1) +
  scale_fill_manual(values   = c("Prior mean = 400 (misspecified)"  = "#D6604D",
                                  "Prior mean = 1000 (near truth)"   = "#2166AC")) +
  scale_colour_manual(values = c("Prior mean = 400 (misspecified)"  = "#D6604D",
                                  "Prior mean = 1000 (near truth)"   = "#2166AC")) +
  facet_wrap(~ prior_label, ncol = 2, scales = "free") +
  labs(
    title    = "Prior Sensitivity Analysis: Posterior of μ",
    subtitle = "Black dashed = true μ = 800  |  Blue = prior mean near truth  |  Red = misspecified prior",
    x        = expression(mu),
    y        = "Density"
  ) +
  theme_portfolio() +
  theme(legend.position = "bottom")

### Interpretation

The results are pretty clear. When the prior is tight and wrongly centred (e.g. $\mathcal{N}(400, 5)$), the posterior gets pulled hard toward 400 and is badly biased. When the prior is vague ($\mathcal{N}(1000, 100)$), the likelihood takes over and the posterior lands close to the true value of 800.

With 500 observations, the data are quite informative, so even mildly misspecified priors with moderate variance ($s = 20$) only cause a small bias. But it's a good reminder that tight priors can really dominate if the data aren't overwhelming — prior choice matters and should be justified.

| Prior | Effect |
|-------|--------|
| N(400, 5) | Badly biased, posterior near 400 |
| N(400, 20) | Partially biased |
| N(1000, 5) | Biased toward 1000 |
| N(1000, 20) | Small bias, likelihood partially wins |
| N(1000, 100) | Posterior close to true value |


---
<a id='2.7'></a>
## 2.7 Key Conclusions

**Part 1 — Mixture Model**

- The mixture model is a sensible approach for RT data where participants might not always be engaged. It explicitly models the "inattentive" component rather than just ignoring outliers.
- The posterior of $\beta$ was positive in this dataset, which goes against the lexical-access hypothesis. But this can't be taken as direct evidence against the hypothesis without a proper Bayes factor comparison.
- Task engagement $\theta \approx 0.71$, roughly consistent with the prior.
- Both IS and MCMC gave similar results, though MCMC is more robust for this kind of model.

**Part 2 — HMC**

- More samples = smoother, more stable posterior estimates. With very few samples the chain hasn't explored enough.
- Step-size needs to be tuned carefully — too small is just as bad as too large because the chain doesn't move enough within the sample budget.
- When the step-size is well-tuned, HMC mixes very efficiently with low autocorrelation.
- With enough data (n=500), vague priors allow the likelihood to dominate. Tight misspecified priors can bias things significantly.
